# Training an anomaly detector

This notebook does the same as `train.py`, step by step. Once you're happy with a setup, use the script
to run longer trainings (see `run-training.sh`).

<!-- TODO(organisers): explain the autoencoder idea: learn to reconstruct normal events, and flag events that reconstruct badly. -->

In [ ]:
import torch
import numpy as np
from tqdm import tqdm

from dataloader import create_dataloaders
from models.registry import REGISTRY
from utils import get_device, count_parameters
import plotting

In [ ]:
device = get_device("auto")  # cuda if available
data_config = "config/baseline.yml"
model_type = "dense-ae"

print("Available models:", list(REGISTRY))
model_cfg = REGISTRY[model_type]
input_type = model_cfg["input_type"]

train_loader, val_loader, _ = create_dataloaders(
    data_config,
    batch_size=1024,
    shuffle=True,
    input_type=input_type,
)
print(f"{len(train_loader.dataset):,} training events, {len(val_loader.dataset):,} validation events, device={device}")

In [ ]:
# The model inputs, after the scaling defined in the config
plotting.plot_data_distributions(train_loader);

In [ ]:
input_shape = tuple(train_loader.dataset.tensors[0].shape[1:])
model = model_cfg["class"](input_shape)

print(model)
print(f"Number of trainable parameters: {count_parameters(model)}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.to(device)

In [ ]:
N_EPOCHS = 5
OUT_PATH = "output"


@torch.no_grad()
def validate():
    model.eval()
    losses = [model(x.to(device), mask.to(device)).cpu() for x, mask in val_loader]
    model.train()
    return torch.cat(losses).mean().item()


history = {"train": [], "val": []}
global_step = 0
for epoch in range(N_EPOCHS):
    running_loss, running_n = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Training epoch {epoch + 1}/{N_EPOCHS}", unit="batch")
    for x, mask in pbar:
        x, mask = x.to(device), mask.to(device)
        optimizer.zero_grad()
        loss = model(x, mask).mean()  # mean anomaly score (= reconstruction error) over the batch
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_n += 1
        global_step += 1
        pbar.set_postfix({"running loss": f"{running_loss / running_n:.4f}"})

    history["train"].append((global_step, running_loss / running_n))
    history["val"].append((global_step, validate()))
    print(f"Epoch {epoch + 1}: val loss {history['val'][-1][1]:.4f}")

plotting.plot_loss_curve(history);

In [ ]:
from pathlib import Path

Path(OUT_PATH, model_type).mkdir(parents=True, exist_ok=True)
torch.save(model, f"{OUT_PATH}/{model_type}/best_model.pth")

## Next steps

<!-- TODO(organisers): what to try next, e.g. other models in models/, different configs, adding your own model. -->

Evaluate your model in [`evaluate.ipynb`](evaluate.ipynb).